In [ ]:
# Libraries Imports
import kagglehub # for downloading the dataset from kaggle
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.dates as mdates
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

plt.style.use('fivethirtyeight')

In [ ]:
# Downloading the dataset
path = kagglehub.dataset_download("vaibhavsxn/google-stock-prices-training-and-test-data")
print("Path to dataset files:", path)

In [ ]:
# Reading the dataset using Pandas
df_train = pd.read_csv(path + "/Google_Stock_Price_Train.csv",parse_dates=["Date"], index_col=["Date"])
df_test = pd.read_csv(path + "/Google_Stock_Price_Test.csv",parse_dates=["Date"], index_col=["Date"])
df_train.head()

# Data Wrangling

In [ ]:
df_train.dtypes # Checking the data types

In [ ]:
df_test.dtypes # Checking the data types

In [ ]:
df_train.sort_index(ascending=True, inplace=True)
df_test.sort_index(ascending=True, inplace=True)

In [ ]:
df_train['Volume'] = pd.to_numeric(df_train['Volume'].str.replace(',', ''))
df_test['Volume'] = pd.to_numeric(df_test['Volume'].str.replace(',', ''))
df_train['Close'] = pd.to_numeric(df_train['Close'].astype(str).str.replace(',', ''))
# Converting the 'Volume' and 'Close' columns to numeric

In [ ]:
df_train.dtypes # Rechecking the data types

In [ ]:
df_test.dtypes # Rechecking the data types

In [ ]:
df_train.isnull().sum() # Checking for null values

In [ ]:
df_test.isnull().sum() # Checking for null values 

In [ ]:
# Plotting the Closing Price
plt.figure(figsize=(16,8))
plt.plot(df_train['Close'], linewidth=1)
plt.title('Google Stock Closing Price over Time')
plt.xlabel('Date')
plt.ylabel('Closing Price')
plt.show()

# Note : The large sudden drop in the stock price is because 
# In April 2014, Google executed a 2-for-1 stock split
# created a new share class (GOOG and GOOGL).

In [ ]:
# Plotting the Volume
plt.figure(figsize=(12,6))
plt.plot(df_train['Volume'], linewidth=1)
plt.title('Google Stock Volume over Time')
plt.xlabel('Date')
plt.ylabel('Volume')
plt.show()

In [ ]:
# Feature Enginerring Moving Average
df_train['MA_3'] = df_train['Close'].rolling(3).mean()
df_train['MA_6'] = df_train['Close'].rolling(6).mean()
df_test['MA_3'] = df_test['Close'].rolling(3).mean()
df_test['MA_6'] = df_test['Close'].rolling(6).mean()

In [ ]:
# Plotting the Moving Averages and close price together

plt.figure(figsize=(14,8))

plt.plot(df_train['Close'], linewidth=1)
plt.plot(df_train['MA_3'], linewidth=1)
plt.plot(df_train['MA_6'], linewidth=1)

plt.legend(['Close', 'MA_3', 'MA_6'])

plt.show()

In [ ]:

df_train["Return"] = df_train["Close"].pct_change()
df_test["Return"] = df_test["Close"].pct_change()

In [ ]:
df_train["HL_Spread"] = df_train["High"] - df_train["Low"]
df_test["HL_Spread"] = df_test["High"] - df_test["Low"]

In [ ]:
df_train["Volume_Change"] = df_train["Volume"].pct_change()
df_test["Volume_Change"] = df_test["Volume"].pct_change()
df_train["Volatility"] = df_train["Close"].rolling(5).std()
df_test["Volatility"] = df_test["Close"].rolling(5).std()

In [ ]:
print(df_train.isna().sum())
print(df_test.isna().sum())

In [ ]:
df_train = df_train.dropna()
df_test = df_test.dropna()
df_train.isnull().sum()

In [ ]:
sns.pairplot(df_train)
plt.show()

In [ ]:
# to check which features should i use for the model based on correlation with close
corr = df_train.corr()

plt.figure(figsize=(12,6))
sns.heatmap(corr, annot=True ,cmap="coolwarm", fmt=".2f")
plt.show()

End of Data Wrangling

In [ ]:
df_test.isnull().sum()

### LSTM Model Building

In [ ]:
# Feature Scaling

features = ['Volume','MA_6','Return',]
# did not take open, high, low, MA_3, Volatility, 'HL_Spread','Volume_Change'
# because they have high correlation with 'Close' 

feature_scaler = MinMaxScaler()
target_scaler = MinMaxScaler()

X_train_scaled = feature_scaler.fit_transform(df_train[features])
X_test_scaled  = feature_scaler.transform(df_test[features])

y_train_scaled = target_scaler.fit_transform(df_train[['Close']])
y_test_scaled  = target_scaler.transform(df_test[['Close']])


In [ ]:
# Sequence Creation 
look_back = 2

X_train, y_train = [], []

for i in range(look_back, len(X_train_scaled)):
    X_train.append(X_train_scaled[i-look_back:i])
    y_train.append(y_train_scaled[i])

X_train = np.array(X_train)
y_train = np.array(y_train)

X_test, y_test = [], []

for i in range(look_back, len(X_test_scaled)):
    X_test.append(X_test_scaled[i-look_back:i])
    y_test.append(y_test_scaled[i])

X_test = np.array(X_test)
y_test = np.array(y_test)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)


In [ ]:
# LSTM Model
model = Sequential()

model.add(LSTM(50, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])))
model.add(Dropout(0.2))
model.add(LSTM(50))
model.add(Dropout(0.2))
model.add(Dense(1))
model.compile(optimizer='adam', loss='mse')

model.summary()


In [ ]:
# Train Model
history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=16,
    validation_split=0.1
)


In [ ]:
# Predictions 
predictions = model.predict(X_test)

predicted_prices = target_scaler.inverse_transform(predictions)
actual_prices = target_scaler.inverse_transform(y_test)


In [ ]:
# ----- RMSE, MAE and R2 Evaluation -----

rmse = round(np.sqrt(mean_squared_error(actual_prices, predicted_prices)),2)
mae = round(mean_absolute_error(actual_prices, predicted_prices),2)
r2 = round(r2_score(actual_prices, predicted_prices),2)

print("RMSE:", rmse)
print("MAE:", mae)
print("R2 Score:", r2)


In [ ]:
# Plot Results
dates = df_test.index[look_back:]

plt.figure(figsize=(12,4))
plt.plot(dates, actual_prices, label="Actual Price", linewidth=1)
plt.plot(dates, predicted_prices, label="Predicted Price", linewidth=1)
plt.title("Stock Price Prediction")
plt.xlabel("Date")
plt.ylabel("Price")
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
plt.xticks(rotation=45)
plt.legend()
plt.show()